# 07 · Harness 实战：内建工具、workspace、context、持久化、telemetry

目标：让 agent 用 SDK **自带** harness 干一件真活，亲眼看到这些能力的价值。

本 notebook 覆盖前几篇没演示的能力：

1. **session.workspace_path** —— 自动分配的工作目录
2. **infinite_sessions** —— 自动 context compaction + 压缩事件
3. **内建工具实战** —— `bash` / `view` / `edit_file` / `grep` / `find_files` / `fetch_url`
4. **available_tools / excluded_tools** —— 全局白/黑名单
5. **reasoning** —— 高 effort 推理 + reasoning delta 流
6. **Session 持久化** —— `list_sessions` / `resume_session` / `delete_session` / `get_last_session_id`
7. **图片输入** —— `view` 工具直接看 jpg
8. **Telemetry** —— OTLP / file 导出，trace context 自动传播

全程用 Azure OpenAI BYOK，不依赖 GitHub 鉴权。

## 0. SDK Harness 全景：你能"白拿"的 6 个能力

这张图把本 notebook 要演示的 6 个 harness 能力按"它们落在系统的哪一层"标出来。如果你正在自己手写其中任何一个，迁到 SDK 等于把那部分代码删掉。

```mermaid
flowchart TB
    subgraph Disk["磁盘 ~/.copilot/"]
        WS["session-state/&lt;session_id&gt;/<br/>① workspace_path<br/>(每个 session 一个独立目录)"]
        Persist["② sessions/<br/>list / resume / delete<br/>历史可重放"]
        Spans["⑥ telemetry/<br/>(可选 file exporter)"]
    end

    subgraph CLI["copilot CLI"]
        Harness["Agent Harness"]
        Compact["③ infinite_sessions<br/>自动 context 压缩<br/>(compaction_start / _complete 事件)"]
        Reason["⑥ reasoning_effort<br/>+ AssistantReasoningDelta"]
        BuiltIn["④ 15 个内建工具<br/>(bash · view · apply_patch · rg ...)"]
        Acl["⑤ available_tools /<br/>excluded_tools<br/>+ permission handler"]
    end

    subgraph App["你的 Python"]
        Code["session.workspace_path<br/>list_sessions / resume_session<br/>SubprocessConfig(telemetry=...)"]
    end

    subgraph Out["上游"]
        OTLP[("OTLP Collector / App Insights")]
    end

    Code --> Harness
    Harness --> WS
    Harness --> Persist
    Harness --> Compact
    Harness --> BuiltIn
    Harness --> Acl
    Harness --> Reason
    Harness -. spans .-> Spans
    Harness -- W3C traceparent --> OTLP
```

| # | SDK 能力 | 替换 cowork_worker 里的什么 |
|---|---|---|
| ① | `session.workspace_path` | 手工 `stage_workspace_manifest()` + `cowork_storage/workspaces/<wid>/` |
| ② | `list_sessions` / `resume_session` / `delete_session` | 自己存 history 到 Cosmos（仍可保留 Cosmos 做元数据索引） |
| ③ | `infinite_sessions` | 自己写 context 截断 / 摘要 |
| ④ | 15 个内建工具 | `worker/tools/{bash,read_file,write_file,edit_file,grep,find_files,fetch_url}.py` |
| ⑤ | `available_tools` / `excluded_tools` + permission handler | Foundry tool allow-list + 自己散落的 if 判断 |
| ⑥ | `SubprocessConfig(telemetry={...})` + reasoning delta | 散落各处的 `enable_instrumentation()` |

> 一句话总结：本 notebook 把这 6 块**一个一个**跑起来给你看——能完整跑通就说明你 `worker/` 目录里同名能力可以**删掉**。


In [ ]:
%pip install -q github-copilot-sdk python-dotenv

In [ ]:
import os, asyncio
from pathlib import Path
from dotenv import load_dotenv
from copilot import CopilotClient, SubprocessConfig
from copilot.session import PermissionHandler
from copilot.generated.session_events import (
    AssistantMessageData, AssistantMessageDeltaData,
    AssistantReasoningData, AssistantReasoningDeltaData,
    SessionIdleData,
)

load_dotenv()

AZURE = {
    'type': 'azure',
    'base_url': os.environ['AZURE_OPENAI_ENDPOINT'],
    'api_key': os.environ['AZURE_OPENAI_API_KEY'],
    'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')},
}
MODEL = os.environ['AZURE_OPENAI_DEPLOYMENT']

async def wait_idle(session) -> str:
    done = asyncio.Event()
    out: list[str] = []
    def on_event(e):
        match e.data:
            case AssistantMessageData() as d:
                out.append(d.content)
            case SessionIdleData():
                done.set()
    unsub = session.on(on_event)
    await done.wait()
    unsub()
    return '\n'.join(out)

## 1. `session.workspace_path` —— 不用自己管目录

默认开启 `infinite_sessions`，SDK 自动给每个 session 分配 `~/.copilot/session-state/{session_id}/`，存 checkpoints 和文件。这就是 cowork_worker 现在手工 `stage_workspace_manifest()` 那段逻辑的替代。

In [ ]:
async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL, provider=AZURE,
    ) as s:
        print('session_id     =', s.session_id)
        print('workspace_path =', s.workspace_path)
        print('exists         =', Path(s.workspace_path).exists())

## 2. 内建工具实战 —— `bash` / `view` / `edit_file` / `grep` / `find_files`

下面让 agent 用 **纯内建工具** 完成一个真任务：

1. 在 workspace 写一个小 Python 包（`bash` + `edit_file`）
2. 用 `grep` 找 TODO
3. 把 TODO 改成实现（`edit_file`）
4. 跑测试（`bash`）

In [ ]:
from copilot.generated.session_events import SessionIdleData

TASK = '''Inside the current directory:
1. Create file calc.py with: def add(a,b): return 0  # TODO: implement
2. Create file test_calc.py with a pytest that asserts add(2,3)==5.
3. Use grep to locate the TODO marker, then edit_file to fix the implementation.
4. Run `python -m pytest test_calc.py -q` via bash and report the result.
Keep edits surgical. Do not ask for confirmation.'''

async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL, provider=AZURE,
    ) as s:
        # 进入它自己的 workspace_path，让所有 bash/edit 都在那
        os.chdir(s.workspace_path)
        def trace(e):
            t = e.type
            if t.startswith('tool.') or t == 'permission.requested':
                print(f'  [{t}] {getattr(e.data, "tool_name", "")}')
        s.on(trace)
        await s.send(TASK)
        print('\n=== FINAL ===')
        print(await wait_idle(s))
        print('\n=== FILES ===')
        for p in sorted(Path(s.workspace_path).glob('*.py')):
            print('---', p.name, '---')
            print(p.read_text())

> 这一段如果跑通，等价于你 `cowork_worker/tools/{bash,edit_file,read_file,grep,find_files,write_file}.py` **整片** 可以删——内建工具行为/性能/权限挂钩都已经齐了。

## 3. `available_tools` / `excluded_tools` —— 全局工具裁剪

- `available_tools=['view', 'grep', 'find_files']` —— 只读 agent
- `excluded_tools=['bash']` —— 禁壳（敏感环境）

黑/白名单是 **session 级**，对所有 sub-agent 也生效。

In [ ]:
async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL, provider=AZURE,
        available_tools=['view', 'grep', 'find_files'],  # 只读
    ) as s:
        await s.send('Try to run `ls /`. If you cannot, say why in one sentence.')
        print(await wait_idle(s))

## 4. Permission handler 按 `kind` 精细裁决

比 `excluded_tools` 更灵活——可以**按路径/命令**裁决。例如：写文件只允许在 `output/`，shell 完全禁。

In [ ]:
from copilot.session import PermissionRequestResult

def policy(req, inv):
    k = req.kind.value
    if k == 'shell':
        return PermissionRequestResult(kind='reject')
    if k == 'write':
        path = req.file_name or ''
        if 'output/' not in path:
            return PermissionRequestResult(kind='reject')
    return PermissionRequestResult(kind='approve-once')

async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=policy, model=MODEL, provider=AZURE,
    ) as s:
        os.chdir(s.workspace_path)
        Path('output').mkdir(exist_ok=True)
        await s.send('Write "hi" to ./output/a.txt, then try to write "hi" to ./b.txt. Report what worked.')
        print(await wait_idle(s))

## 5. Infinite sessions —— 自动 context 压缩

默认开启。可调阈值：达到 `background_compaction_threshold` 后台压缩，到 `buffer_exhaustion_threshold` 前台阻塞直到压缩完。

事件：`session.compaction_start` / `session.compaction_complete`。

In [ ]:
async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL, provider=AZURE,
        infinite_sessions={
            'enabled': True,
            'background_compaction_threshold': 0.50,  # 半满就压（仅演示）
            'buffer_exhaustion_threshold': 0.90,
        },
    ) as s:
        wsp = Path(s.workspace_path)
        events_seen: list[tuple[str, dict]] = []

        def tap(e):
            t = e.type
            if 'compaction' in t:
                # dump 完整 payload 用来还原 schema
                data = getattr(e.data, '__dict__', {}) or {}
                events_seen.append((t, dict(data)))
                print(f'  ↪ {t}  payload_keys={list(data.keys())}')
        s.on(tap)

        # 喂一些长 prompt 凑 context
        for i in range(3):
            await s.send(f'Round {i}: list 30 random English words, each on its own line.')
            await wait_idle(s)

        # ---- 探针 1：内存里的消息历史 ----
        msgs = await s.get_messages()
        print(f'\n[probe] get_messages() returned {len(msgs)} events')
        kinds: dict[str, int] = {}
        for m in msgs:
            k = getattr(m, 'type', type(m).__name__)
            kinds[k] = kinds.get(k, 0) + 1
        for k, v in sorted(kinds.items(), key=lambda x: -x[1]):
            print(f'   {v:>4}  {k}')

        # ---- 探针 2：workspace_path 目录里磁盘上有什么 ----
        print(f'\n[probe] ls {wsp}')
        for p in sorted(wsp.rglob('*')):
            if p.is_file():
                size = p.stat().st_size
                print(f'   {size:>8}  {p.relative_to(wsp)}')

        # ---- 探针 3：完整 compaction 事件 payload ----
        if events_seen:
            print('\n[probe] compaction events full payloads:')
            for t, data in events_seen:
                print(f'   {t}:')
                for k, v in data.items():
                    s_ = repr(v)
                    print(f'      {k} = {s_[:200]}{"..." if len(s_)>200 else ""}')
        else:
            print('\n[probe] no compaction events fired — try lowering threshold further '
                  'or sending longer prompts')

## 6. Reasoning 模型 —— `reasoning_effort` + reasoning delta

对支持的模型（o-系列、Claude reasoning 模式等）开启 `reasoning_effort='high'`，并订阅 `AssistantReasoningDeltaData` 拿到思考链。

In [ ]:
async def run_reasoning(prompt: str, model: str = MODEL):
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=model, provider=AZURE,
            reasoning_effort='high', streaming=True,
        ) as s:
            done = asyncio.Event()
            def on_event(e):
                match e.data:
                    case AssistantReasoningDeltaData() as d:
                        print('🧠', d.delta_content, end='', flush=True)
                    case AssistantMessageDeltaData() as d:
                        print(d.delta_content or '', end='', flush=True)
                    case SessionIdleData():
                        done.set()
            s.on(on_event)
            await s.send(prompt)
            await done.wait()

# 模型不支持 reasoning 时 SDK 会静默忽略 reasoning_effort
await run_reasoning('A farmer has 17 sheep, all but 9 die. How many are left? Show your reasoning briefly.')

## 7. Session 持久化 —— list / resume / delete

Session 状态默认落盘到 `~/.copilot/session-state/`。重启后可以 `resume_session(id)` 接着聊，history + workspace 文件都还在。这是 cowork_worker 现在 **没有** 的能力——你目前依赖 Cosmos 自存。

In [ ]:
async with CopilotClient() as client:
    # 第 1 轮
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all, model=MODEL, provider=AZURE,
    ) as s:
        sid = s.session_id
        await s.send('My favorite color is teal. Remember it.')
        await wait_idle(s)

    # 列出 + 找到刚才那个
    sessions = await client.list_sessions()
    print('total sessions on disk:', len(sessions))
    last = await client.get_last_session_id()
    print('last session id      :', last)

    # 第 2 轮 —— resume
    async with await client.resume_session(
        sid, on_permission_request=PermissionHandler.approve_all,
    ) as s2:
        await s2.send('What is my favorite color?')
        print('RESUMED →', await wait_idle(s2))

    # 清理
    await client.delete_session(sid)
    print('deleted', sid)

## 8. 图片输入 —— `view` 工具直接读盘 + attachment 推送

两种姿势：
1. **File attachment**：让 runtime 自己读 `path`
2. **Blob attachment**：base64 直推（适合后端转发用户上传，不落盘）
3. 已落盘的话，直接让模型用内建 `view` 工具看

In [ ]:
# 示例骨架（需有一张本地 jpg）
# await s.send(
#     'What is in this image?',
#     attachments=[{'type': 'file', 'path': '/tmp/cat.jpg'}],
# )
# 或：
# await s.send('View the most recent jpg in the current directory and describe it.')

## 9. Telemetry —— OTLP 导出 + 自动 trace 传播

把 `telemetry` 配在 `SubprocessConfig` 上，CLI 就会按 W3C Trace Context 把 traceparent/tracestate **自动**在 SDK ↔ CLI ↔ 你的 tool handler 之间传播。

- `otlp_endpoint='http://localhost:4318'` —— 推到 OTEL Collector / App Insights Bridge
- `file_path='/tmp/spans.jsonl'` + `exporter_type='file'` —— 本地排查

In [ ]:
# 演示：导到文件，跑一轮，看 span
trace_file = '/tmp/copilot_spans.jsonl'
Path(trace_file).unlink(missing_ok=True)

cfg = SubprocessConfig(telemetry={
    'exporter_type': 'file',
    'file_path': trace_file,
    'capture_content': True,
    'source_name': 'cowork-sdk-poc',
})

async with CopilotClient(cfg) as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL, provider=AZURE,
    ) as s:
        await s.send('Say hi.')
        await wait_idle(s)

import json
with open(trace_file) as f:
    for i, line in enumerate(f):
        if i >= 3: break
        span = json.loads(line)
        print(span.get('name'), '→', span.get('attributes', {}).get('gen_ai.operation.name', ''))

## 10. Takeaways for cowork_worker

| 现在自己做 | SDK 内建替代 |
|---|---|
| `worker/tools/{bash,read_file,write_file,edit_file,grep,find_files,fetch_url}.py` | 全部内建工具，零代码 |
| `stage_workspace_manifest()` + 自己管 `cowork_storage/workspaces/<wid>/` | `session.workspace_path` |
| 自己实现 context 截断 / 摘要 | `infinite_sessions` 自动压缩 + 事件可观测 |
| 自己存 session history 到 Cosmos | `list_sessions` / `resume_session` 直接给 |
| `enable_instrumentation()` 散落各处 | `SubprocessConfig(telemetry=...)` 一行 |
| 工具/权限分两层（Foundry allow-list + 自己 if 判断） | `available_tools` + `on_permission_request` 一处搞定 |

**唯一要保留的自定义工具**：`python_tool`（rlimit 沙箱 + LLM 专用 venv）—— SDK `bash` 不带这层。

下一篇 (08) 演示 **skills / MCP / custom agents / slash commands**，对应你 `SkillsContextProvider` + 多 agent 编排的需求。